# Step 1: Data Exploration, Stratified Sampling & Preprocessing Pipeline

**Capstone Project: IoT Intrusion Detection System (CICIoT2023)**

This notebook demonstrates:
1. Loading the stratified working dataset extracted from `MERGED_CSV/`.
2. Analyzing and visualizing class distributions across **2 classes (Binary)**, **8 classes (Category)**, and **34 classes (Fine-Grained)**.
3. Inspecting the 39 network flow features and verifying absence of missing values.
4. Applying the leak-free preprocessing and `StandardScaler` pipeline.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (
    SAMPLE_FILE,
    FEATURE_COLUMNS,
    TARGET_COLUMN,
    LABEL_MAPPING_BINARY,
    LABEL_MAPPING_8CLASSES,
    BINARY_CLASSES,
    EIGHT_CLASSES,
    THIRTY_FOUR_CLASSES,
)
from src.preprocessing import prepare_dataset, encode_labels, clean_features

sns.set_theme(style="whitegrid", palette="muted")
print(f"Target sample path: {SAMPLE_FILE}")

## 1. Load the Stratified Sample

In [ ]:
df = pd.read_csv(SAMPLE_FILE)
print(f"Sample Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Total Unique Classes: {df[TARGET_COLUMN].nunique()} of 34")
df.head()

## 2. Target Distribution Visualizations

In [ ]:
# 2-Class Target (Binary)
y_bin, _, bin_classes = encode_labels(df, target_type="binary")
df_bin = pd.Series(y_bin).map({0: "Benign", 1: "Attack"}).value_counts()

# 8-Class Target (Category)
y_cat, _, cat_classes = encode_labels(df, target_type="category")
df_cat = pd.Series(y_cat).map({i: c for i, c in enumerate(cat_classes)}).value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Binary plot
sns.barplot(x=df_bin.values, y=df_bin.index, ax=axes[0], palette=["#2ecc71", "#e74c3c"])
axes[0].set_title("Phase 1: Binary Class Distribution (2 Classes)", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Instance Count")
for i, v in enumerate(df_bin.values):
    pct = (v / len(df)) * 100
    axes[0].text(v + 1000, i, f"{v:,} ({pct:.1f}%)", va="center", fontweight="bold")

# 8-Class plot
sns.barplot(x=df_cat.values, y=df_cat.index, ax=axes[1], palette="viridis")
axes[1].set_title("Phase 2: Functional Category Distribution (8 Classes)", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Instance Count")
for i, v in enumerate(df_cat.values):
    pct = (v / len(df)) * 100
    axes[1].text(v + 1000, i, f"{v:,} ({pct:.1f}%)", va="center", fontsize=9)

plt.tight_layout()
plt.show()

### 34-Class Fine-Grained Target Distribution

In [ ]:
plt.figure(figsize=(12, 10))
class_counts = df[TARGET_COLUMN].value_counts()
sns.barplot(x=class_counts.values, y=class_counts.index, palette="mako")
plt.title("Phase 3: Fine-Grained Attack Distribution (34 Classes)", fontsize=14, fontweight="bold")
plt.xlabel("Instance Count")
plt.ylabel("Attack Profile")
for i, v in enumerate(class_counts.values):
    plt.text(v + 200, i, f"{v:,}", va="center", fontsize=8)
plt.tight_layout()
plt.show()

## 3. Feature Verification and Integrity Check

In [ ]:
print(f"Verifying {len(FEATURE_COLUMNS)} input features:")
missing_counts = df[FEATURE_COLUMNS].isnull().sum()
print(f"Missing values across all features: {missing_counts.sum()}")

# Summary statistics of first 8 features
df[FEATURE_COLUMNS[:8]].describe().T[['count', 'mean', 'std', 'min', 'max']]

## 4. Test Preprocessing Pipeline (Train/Test Split & Scaling)

In [ ]:
# Demonstrate pipeline for Binary classification
X_train, X_test, y_train, y_test, class_names, scaler = prepare_dataset(
    df, target_type="binary", test_size=0.2, random_state=42, scale_features=True
)

print(f"X_train Shape : {X_train.shape} | y_train Shape: {y_train.shape}")
print(f"X_test Shape  : {X_test.shape}  | y_test Shape : {y_test.shape}")
print(f"Classes: {class_names}")
print(f"Train Label Distribution: {dict(pd.Series(y_train).value_counts())}")
print(f"Test Label Distribution : {dict(pd.Series(y_test).value_counts())}")

# Verify normalization: Mean should be ~0 and Std ~1
print(f"\nX_train Mean (first 5 features): {np.round(X_train[:, :5].mean(axis=0), 4)}")
print(f"X_train Std  (first 5 features): {np.round(X_train[:, :5].std(axis=0), 4)}")